# Dunder (Magic) Methods


# __ dict __

Every Python class has a `__ dict __`. Python provides it to instances separately which means main class has its own `__ dict __ ` and each instances have separate `__ dict __`. 

Unlike C++, Python doesn't ship everything to instance that means instances will have no attributes, methods of the class. All are attached to the main class. And when an instance calls a method or attribute then python resolves `__ dict __` from instance to main class's `__ dict __`.

In [1]:
class Dog:
    def bark(self):
        print("woof")


d1 = Dog()
d2 = Dog()

# Dog.__dict__ is main class dictionary
print(f"{Dog.__dict__=}")

# d1.__dict__: dictionary {} belongs to d1 only and d2 has its separate. If we define attributes then each instance will have its own arbitrary attributes.

# Verification: d1 and d2 are now having arbitrary attribute but different to each let's now verify it if the each __dict__ is unique to instance or not.

# Note: these arbitrary attributes are not Dog class part but specific to the instances __dict__.
d1.name = "Rex"
d2.isBark = True

print(f"{d1.__dict__=}")
print(f"{d2.__dict__=}")

Dog.__dict__=mappingproxy({'__module__': '__main__', '__firstlineno__': 1, 'bark': <function Dog.bark at 0x000001680020DF30>, '__static_attributes__': (), '__dict__': <attribute '__dict__' of 'Dog' objects>, '__weakref__': <attribute '__weakref__' of 'Dog' objects>, '__doc__': None})
d1.__dict__={'name': 'Rex'}
d2.__dict__={'isBark': True}


## Output verification:

As mentioned in the comments, each instance has unique `__ dict __` so their outputs are identical. d1 has its own attributes, while d2 owns its own and Global `Dog.__ dict __` has all info.

<br>

---

<br>

## Verify attributes

Look below, attributes are unique to each instance. They are not shared. And we tried to access isBark on d1 and name on d2 so we got the following AttributeError which was expected.

In [3]:
print(d1.isBark)

AttributeError: 'Dog' object has no attribute 'isBark'

In [4]:
print(d2.name)

AttributeError: 'Dog' object has no attribute 'name'

<br>

---

<br>

## Class attributes

In [6]:
# suppose we add attribute to the class
Dog.species = "Canine"  # here, species belong to the class not instances. But, both can access it and Python will resolve by first looking at instances and then Class.

```txt
Current view of class after getting species attribute.

       Dog
------------------
species = "Canine"
bark()

        |
   -------------
   |           |
 d1           d2
```

In [7]:
print(d1.species)
print(d2.species)

Canine
Canine


<br>

---

<br>

## Instance attributes override class attributes

In [8]:
class Dog:
    species = "Canine"


d = Dog()

print(d.species)

Canine


##### The instance gets its own species attribute.

In [9]:
# Now let's update species
d.species = "Wolf"

```txt
Visualize change, class still has its own species saved, while d has got new species saved in its __dict__.

       Dog
------------------
species = "Canine"

        |
        v

        d
------------------
species = "Wolf"
```

### Verify the Change

In [10]:
print(d.species)

Wolf


In [11]:
print(Dog.species)

Canine


### Workflow truth

> Python finds the instance attribute first, so it "shadows" the class attribute.

Next, read the [`02__init__.ipynb`](./02__init__.ipynb)

# __ init __ 

> Please read [01__dict__.ipynb](./01__dict__.ipynb) before reading this notebook file.

Have you ever noticed that `__ init __` is not Python's real constructor. It's just same like `__ dict __` (adding attributes using instance). But, syntax is different and using `__ init __` attributes are directly declared inside the class. That's why oftenly it is known as a the Python's constructor, which is false. 

The truth is; when you write `d1 = Dog("Rex")`, Python actually runs a two-step process bts.

**Step 1: Creation (`__ new __`)** 

Python calls a hidden method called `__ new __` and this is the true constructor. It carves out the memory for the instance, creates the empty `__ dict __` (unless `__ slots __` is used), and wires up the fallback to the class.

**Step 2: Initialization (`__ init __`)**

Once the empty shell exists, Python automatically passes it to `__ init __` as self parameter.

When you write self.name = name inside `__ init __`, Python simply looks at the self object, accesses its `__ dict __`, and inserts the key-value pair.

<br>

---

<br>

#### These two blocks of code result in the exact same __dict__ state:

In [3]:
# Approach 1: Using __init__
class Dog:
    def __init__(self, name):
        self.name = name  # Mutating the __dict__ from inside


d1 = Dog("Rex")

In [4]:
# Approach 2: Doing it manually
class Dog:
    pass


d2 = Dog()
d2.name = "Rex"  # Mutating the __dict__ from outside

> In both cases, `d1.__ dict __` and `d2.__ dict __` will equal {'name': 'Rex'}.

<br>

---

<br>

### Why Do We Use `__ init __` Then?

If it does not change the underlying mechanics, why is it a cornerstone of Python design?

1. **Standardized Contracts**: __init__ guarantees that an object is "born" with all the necessary state to function. If your bark method relies on self.volume, putting self.volume = 10 in __init__ ensures bark() will never crash with an AttributeError.

2. **Preventing Spaghetti Code**: If we did not have __init__, developers would have to manually add attributes scattered throughout the codebase after creating the object. __init__ centralizes the object's definition so anyone reading the code immediately knows exactly what data the instance manages.

3. **Immutability (Kind of)**: While Python doesn't enforce strict immutability by default, requiring parameters in __init__ (like d1 = Dog("Rex")) makes it structurally clear that certain data is foundational to the object's identity from the moment it is created.

Next, read the [03__slots__.ipynb](03__slots__.ipynb) to learn how do disable/destroy `__ dict __` and save huge memory.

# __ slots __

> Please read [02__init__.ipynb](./02__init__.ipynb) before reading this notebook file.

Let's first understand: **why is `__ dict __` both good and bad?**

- **Why it is good**: A `__ dict __` gives instances dynamic flexibility. You can do `d1.isBark = True` on the fly.

- **Why it is heavy**: Under the hood in CPython, a dictionary is a hash table. Hash tables are incredibly fast for lookups, but they are memory hogs. They require allocating extra empty space (to avoid hash collisions) and store a lot of metadata. If you have 1,000,000 Dog instances, you have 1,000,000 bloated hash tables in your RAM.

<br>

---

<br>

# Enter __ slots __: The memory Diet

When you define `__ slots __` (usually by defining a list or tuple inside the class body), Python completely deletes the `__ dict __` from the instance. Instead of creating a heavy hash table, Python allocates a tiny, fixed-size C-style array (a struct) for the instance in memory.

In simple words, `__ slots __` are used to prevent users from adding arbitrary, unintended attributes to a Python class instance.

By default, Python uses a dictionary (`__ dict __`) to store instance variables, allowing anyone to add new ones dynamically. Defining `__ slots __` replaces `__ dict __` with a fixed-size array, blocking arbitrary assignments.

<br>

---

<br>

# Code Example using __ slots __

In [9]:
# Manual method can be used with tuples __slots__ = ("name", "age")


class Person:
    # Explicitly list allowed attributes as strings
    __slots__ = ["name", "age"]  # list contains attributes of the class

    def __init__(self, name, age):
        self.name = name
        self.age = age


p1 = Person("Saqib", 22)

#### Verify p1.__ dict __

In [10]:
print(p1.__dict__)

AttributeError: 'Person' object has no attribute '__dict__'

#### Interpretation

You saw, how just by adding a single line of list starting with `__ slots __` deletes `__ dict __` on instance and we got the error. So, no more any instance of class Person has `__ dict __`.

<br>

---

<br>

# Another Implementation Method: The @dataclass(slots=True) Decorator

Introduced in Python 3.10, this is the modern standard for writing slotted classes. Introduced in Python 3.10, this is the modern standard for writing slotted classes. It automatically generates the __slots__ tuple behind the scenes based on your type hints.

### Code Example

@dataclass decorator automatically writes `__ init __` uth (under the hood). It reads your type hints and builds the constructor automatically, saving you from writing boilerplate code.

In [11]:
from dataclasses import dataclass


@dataclass(slots=True)
class Person:
    name: str
    age: int
    height: float


p1 = Person("Saqib", 22, 5.4)

Python automatically generates and executes code equivalent to this:

```python
class Person:
    __slots__ = ("name", "age", "height")

    def __init__(self, name: str, age: int, height: float)
        self.name = name
        self.age = age
        self.height = height
```

<br>

---

<br>

# Verification `__ dict __` and arbitrary attribute insertion

In [12]:
print(p1.__dict__)

AttributeError: 'Person' object has no attribute '__dict__'

## verify arbitrary attribute insertion

In [13]:
p1.country = "Pakistan"

AttributeError: 'Person' object has no attribute 'country' and no __dict__ for setting new attributes

> Both, manual `__ slots __` and `decorator __ slots __` methods, delete `__ dict __` on class instances.

<br>

---

<br>

# `__repr__` ("representation") Magic Method or  dunder (double underscore) Method

It is a special built-in method in Python used to define the official string representation of an object.

Its primary purpose is to be unambiguous and helpful for developers during debugging and logging. Ideally, the string returned by `__repr__` should look like a valid Python expression that you could copy and paste to recreate that exact same object.

### Without `__repr__`

In [2]:
class Student:
    # constructor
    def __init__(self, id, name, dept):
        self.id = id
        self.name = name
        self.dept = dept

    # iterable
    def __iter__(self):
        yield self.id
        yield self.name
        yield self.dept


import uuid

s1 = Student(str(uuid.uuid4()), "Saqib Bedar", "CS")

# print s1 and see the output, without __repr__ Python will output a cryptic memory address <__main__.Student object at 0x000002CBDA8B56A0>
print(s1)  # Output: <__main__.Student object at 0x000002CBDA8B56A0>

### With `__repr`

In [ ]:
class Student:
    def __init__(self, id, name, dept):
        self.id = id
        self.name = name
        self.dept = dept

    # The official developer representation
    def __repr__(self):
        return f"Student(id={self.id!r}, name={self.name!r}, dept={self.dept!r})"

    # Iterable
    def __iter__(self):
        yield self.id
        yield self.name
        yield self.dept


import uuid

# create an object
std = Student(str(uuid.uuid4()), "Saqib Bedar", "CS")

# Triggers __repr__
print(std)

Student(id='16b1b46d-985e-4719-844d-4e36651a9227', name='Saqib Bedar', dept='CS')


> Note: The `!r` flag inside the f-string forces Python to call `__repr__` on those individual attributes, which automatically wraps strings like name and department in quotes for clarity.

---

# `__str__` dunder Method

The `__str__` magic method (or "dunder" method) defines a human-readable, user-friendly string representation of a class object.

When you pass an object to the `print()` function, `str()`, or an `f-string`, Python automatically calls the class's `__str__` method behind the scenes.

In [1]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    # Used for clean human readable display. Primary goal: end-user
    def __str__(self):
        return f"'{self.title}' by {self.author}"


# Create a new book (instance of a book)
my_book = Book("Fluent Python", "Saqib Bedar")

# trigger __str__ automatically
print(my_book)

'Fluent Python' by Saqib Bedar


### 3 Rules for Using `__str__`

- **Must return a string**: The method cannot return `integers`, `lists`, or `None`. If it returns anything other than a `str` `type`, Python raises a `TypeError`.

- Do not call `print()` inside it: The method should only construct and return the string. Let the external code handle how to print or display it.

- Accepts only `self`: You cannot pass additional parameters into the `__str__(self)` definition.

---

# `__repr__` vs `__str__`

Let' now cut the confusion about both as both are returning `str`, and on `print()` we have different output, too. But, who will get the precedence when both are defined and how Python will resolve it. Let's first see goal of each once again.

- `__repr__`: Aimed at **developers**. Used for debugging, logging, and interactive shells. It must be detailed and unambiguous.

- `__str__`: Aimed at **end-users**. Used when you want a clean, human-readable display (like on a web page or user interface).

### Priority

When both methods are defined, `print(my_book)` will always prioritize `__str__`. 

Python looks for the user-friendly representation first when outputting to the console via `print()`.

In [2]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    # priority: 1
    def __str__(self):
        return f"'{self.title}' by {self.author}"

    # priority: 2 (or fallback)
    def __repr__(self):
        return f"Book('{self.title!r}, {self.author!r}')"


my_book = Book("Fluent Python", "Saqib Bedar")

# This triggers __str__
print(my_book)

'Fluent Python' by Saqib Bedar


### Fallback Behavior

If you only define `__repr__`, Python will use it as a backup for `print()`.

- `__str__` falls back to `__repr__` if `__str__` is missing.

- `__repr__` never falls back to `__str__`.